In [40]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

import pickle

# =========================
# 2. LOAD DATA (.xlsx)
# =========================

df = pd.read_excel(r"D:/fake_news_detection/bharatfakenewskosh (3).xlsx")

df.head()

# =========================
# 3. HANDLE MISSING VALUES
# =========================
df = df.fillna("")

# =========================
# 4. FIX LABEL COLUMN
# =========================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].map({'TRUE': 1, 'FALSE': 0})
df = df.dropna(subset=['Label'])

# =========================
# 5. DOWNLOAD NLTK
# =========================
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# =========================
# 6. CLEAN TEXT FUNCTION
# =========================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]

    return " ".join(tokens)

# =========================
# 7. COMBINE TEXT
# =========================
df['content'] = df['Eng_Trans_Statement'] + " " + df['Eng_Trans_News_Body']
df['content'] = df['content'].apply(clean_text)

# =========================
# 8. FEATURES & TARGET
# =========================
X = df['content']
y = df['Label']

# =========================
# 9. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# 10. PIPELINE (TF-IDF + LOGISTIC REGRESSION)
# =========================
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=12000,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9
    )),
    ("model", LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        C=2
    ))
])

# =========================
# 11. CROSS VALIDATION
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1')

print("Cross-validation F1:", scores)
print("Mean F1:", scores.mean())

# =========================
# 12. TRAIN MODEL
# =========================
pipeline.fit(X_train, y_train)

# =========================
# 13. EVALUATION
# =========================
y_pred = pipeline.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# =========================
# 14. PREDICTION DISTRIBUTION 
# =========================
print("\nPrediction distribution:")
print(pd.Series(y_pred).value_counts())

# =========================
# 15. FINAL PREDICTION FUNCTION 
# =========================
def predict_news(text):
    text = clean_text(text)
    
    prob = pipeline.predict_proba([text])[0]
    prob_fake = prob[1]
    
    # Strict binary decision
    if prob_fake >= 0.55:   # tuned threshold to avoid "all fake"
        return f"🚨 Fake News (Confidence: {prob_fake:.2f})"
    else:
        return f"✅ Real News (Confidence: {1 - prob_fake:.2f})"

# =========================
# 16. TEST SAMPLE
# =========================
sample = "Government launches new education policy across India"
print("\nSample Prediction:", predict_news(sample))

# =========================
# 17. SAVE MODEL
# =========================
pickle.dump(pipeline, open("model1.pkl", "wb"))

print("\n✅ Model saved successfully!")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Gourav\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Gourav\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Cross-validation F1: [0.59422059 0.60468943 0.60824954 0.61041074 0.58579144]
Mean F1: 0.6006723476823406

Accuracy: 0.5443110348770727

Classification Report:
               precision    recall  f1-score   support

           0       0.43      0.48      0.45      2064
           1       0.63      0.59      0.61      3183

    accuracy                           0.54      5247
   macro avg       0.53      0.53      0.53      5247
weighted avg       0.55      0.54      0.55      5247


Prediction distribution:
1    2948
0    2299
Name: count, dtype: int64

Sample Prediction: ✅ Real News (Confidence: 0.58)

✅ Model saved successfully!


In [ ]:
sample = "The Government of India has launched a new National Education Policy aimed at improving the quality of education, promoting digital learning, and increasing access to higher education across the country."
print("\nSample Prediction:", predict_news(sample))

In [55]:
print(predict_news("The Ministry of Health launched a nationwide vaccination drive to prevent seasonal diseases."))

🚨 Fake News (Confidence: 0.84)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Gourav\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Gourav\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Cross-validation F1: [0.67114094 0.67733139 0.66951514 0.68550593 0.68468468]
Mean F1: 0.6776356168161987

Accuracy: 0.584715075281113

Classification Report:
               precision    recall  f1-score   support

           0       0.46      0.33      0.38      2064
           1       0.63      0.75      0.69      3183

    accuracy                           0.58      5247
   macro avg       0.55      0.54      0.54      5247
weighted avg       0.57      0.58      0.57      5247


Prediction distribution:
1    3772
0    1475
Name: count, dtype: int64

Sample Prediction: 🚨 Fake News (Confidence: 0.52)

✅ Model saved successfully!


In [5]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

from sklearn.utils import resample

import pickle

# =========================
# 2. LOAD DATA
# =========================
df = pd.read_excel(r"D:/fake_news_detection/bharatfakenewskosh (3).xlsx")
df = df.fillna("")

# =========================
# 3. FIX LABEL
# =========================
df['Label'] = df['Label'].astype(str).str.strip().str.upper()
df['Label'] = df['Label'].map({'TRUE': 1, 'FALSE': 0})
df = df.dropna(subset=['Label'])

# =========================
# 4. BALANCE DATASET 
# =========================
df_fake = df[df['Label'] == 1]
df_real = df[df['Label'] == 0]

# Upsample REAL
df_real_upsampled = resample(
    df_real,
    replace=True,
    n_samples=len(df_fake),
    random_state=42
)

df = pd.concat([df_fake, df_real_upsampled])

print("Balanced data:")
print(df['Label'].value_counts())

# =========================
# 5. TEXT CLEANING
# =========================
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(tokens)

# =========================
# 6. FEATURE CREATION
# =========================
df['content'] = (
    df['Eng_Trans_Statement'] + " " +
    df['Eng_Trans_News_Body'] + " " +
    df['News_Category'] + " " +
    df['Platform']
)

df['content'] = df['content'].apply(clean_text)

X = df['content']
y = df['Label']

# =========================
# 7. SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# 8. MODEL
# =========================
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9
    )),
    ("model", LogisticRegression(max_iter=2000))
])

# =========================
# 9. TRAIN
# =========================
pipeline.fit(X_train, y_train)

# =========================
# 10. EVALUATE
# =========================
y_pred = pipeline.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

print("\nPrediction distribution:")
print(pd.Series(y_pred).value_counts())

# =========================
# 11. PREDICTION FUNCTION
# =========================
def predict_news(text):
    text = clean_text(text)
    prob = pipeline.predict_proba([text])[0]
    prob_fake = prob[1]

    if prob_fake >= 0.5:
        return f"🚨 Fake News ({prob_fake:.2f})"
    else:
        return f"✅ Real News ({1 - prob_fake:.2f})"

# =========================
# 12. TEST
# =========================
print(predict_news("India successfully launched a new satellite."))
print(predict_news("Government will give 10 lakh rupees to everyone tomorrow."))

# =========================
# 13. SAVE
# =========================
pickle.dump(pipeline, open("final_model.pkl", "wb"))

Balanced data:
Label
1    15913
0    15913
Name: count, dtype: int64


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Gourav\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Gourav\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Accuracy: 0.649387370405278

Report:
               precision    recall  f1-score   support

           0       0.65      0.66      0.65      3183
           1       0.65      0.64      0.65      3183

    accuracy                           0.65      6366
   macro avg       0.65      0.65      0.65      6366
weighted avg       0.65      0.65      0.65      6366


Prediction distribution:
0    3227
1    3139
Name: count, dtype: int64
✅ Real News (0.55)
🚨 Fake News (0.58)


In [6]:
# =========================
# TEST MULTIPLE NEWS SAMPLES
# =========================

test_samples = [
    # ✅ Real news
    "India successfully launched a new satellite to improve communication services.",
    "The Reserve Bank of India has revised interest rates to control inflation.",
    "The government introduced a new policy to promote digital education in rural areas.",

    # 🚨 Fake-like news
    "Breaking: Government secretly plans to shut down the internet across India next month.",
    "Scientists confirm that drinking hot water cures all diseases instantly.",
    "Government will deposit 10 lakh rupees in every citizen's bank account tomorrow."
]

# Run predictions
for i, news in enumerate(test_samples, 1):
    print(f"\nTest Case {i}:")
    print("News:", news)
    print("Prediction:", predict_news(news))


Test Case 1:
News: India successfully launched a new satellite to improve communication services.
Prediction: 🚨 Fake News (0.51)

Test Case 2:
News: The Reserve Bank of India has revised interest rates to control inflation.
Prediction: 🚨 Fake News (0.66)

Test Case 3:
News: The government introduced a new policy to promote digital education in rural areas.
Prediction: ✅ Real News (0.50)

Test Case 4:
News: Breaking: Government secretly plans to shut down the internet across India next month.
Prediction: 🚨 Fake News (0.57)

Test Case 5:
News: Scientists confirm that drinking hot water cures all diseases instantly.
Prediction: 🚨 Fake News (0.58)

Test Case 6:
News: Government will deposit 10 lakh rupees in every citizen's bank account tomorrow.
Prediction: 🚨 Fake News (0.54)
